# EDA original (referencia, no es parte de producción)

Notebook exploratorio con el que empezó el proyecto: GARCH(1,1)-t sobre SPY (pruebas ADF, Ljung-Box, ARCH-LM, Monte Carlo, VaR/CVaR) y el detector de anomalías con menú interactivo. La versión automatizada vive en `src/` y se ejecuta con `run_pipeline.py`. Se limpiaron las salidas y una línea suelta (`message.txt`) al final de la celda del menú que provocaba `NameError`.

In [ ]:
pip install yfinance matplotlib pandas statsmodels arch

In [ ]:
# -*- coding: utf-8 -*-
"""
Script mejorado para modelado GARCH(1,1) con distribución t + simulación Monte Carlo GARCH
Incluye tests: ADF, Ljung-Box, ARCH LM; validación fuera de muestra; métricas AIC/BIC; VaR/CVaR.
Fecha: (ejecútalo para actualizar)
"""

import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from arch import arch_model
from scipy.stats import entropy
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
from scipy.stats import t as student_t

# === Parámetros principales ===
ticker = "SPY"
start_date = "2015-01-01"
n_days = 5            # horizonte de forecast en días hábiles
n_sim = 10000         # simulaciones Monte Carlo
n_plot_sim = 100      # cuántas trayectorias graficar (subset)
oos_days = 60         # tamaño de validación fuera de muestra (días)

# === 1) Descargar datos ===
df = yf.download(ticker, start=start_date, progress=False)
prices = df['Close'].dropna()

# Opcional: recortar si hay pocos datos
if len(prices) < 300:
    raise ValueError("No hay suficientes datos para un análisis robusto.")

# === 2) Calcular rendimientos logarítmicos diarios ===
returns = np.log(prices / prices.shift(1)).dropna()
returns.name = 'returns'

# === 3) Gráficas iniciales (preliminares) ===
plt.figure(figsize=(12, 4))
plt.plot(prices, label=f'Precio {ticker} (Close)')
plt.title(f'{ticker} - Precio de cierre')
plt.xlabel('Fecha'); plt.ylabel('Precio'); plt.grid(True); plt.legend()
plt.show()

plt.figure(figsize=(12, 3))
plt.plot(returns, label='Rendimientos log')
plt.title(f'{ticker} - Rendimientos logarítmicos diarios')
plt.xlabel('Fecha'); plt.ylabel('Rendimiento'); plt.grid(True); plt.legend()
plt.show()

plt.figure(figsize=(12, 3))
plot_acf(returns, lags=30, ax=plt.gca())
plt.title('ACF de rendimientos'); plt.grid(True); plt.show()

plt.figure(figsize=(12, 3))
plot_pacf(returns, lags=30, ax=plt.gca(), method='ywm')
plt.title('PACF de rendimientos'); plt.grid(True); plt.show()

# === 4) Tests estadísticos preliminares ===
print("\n---- Tests preliminares ----")
adf_res = adfuller(returns)
print(f"ADF: stat={adf_res[0]:.4f}, p-val={adf_res[1]:.4f} (H0: raíz unitaria)")

# Ljung-Box en retornos y en retornos al cuadrado
lb_ret = acorr_ljungbox(returns, lags=[10], return_df=True)
lb_ret2 = acorr_ljungbox(returns**2, lags=[10], return_df=True)
print("\nLjung-Box (retornos) lag=10:\n", lb_ret)
print("\nLjung-Box (retornos^2) lag=10 (test de autocorrelación en varianza):\n", lb_ret2)

# ARCH LM test (usando het_arch)
arch_test = het_arch(returns, nlags=10)
print("\nARCH LM test (F-statistic, p-value):", arch_test[0], arch_test[1])

# === 5) Split train / out-of-sample (validación) ===
train = returns.iloc[:-oos_days]
oos = returns.iloc[-oos_days:]

print(f"\nTamaño entrenamiento: {len(train)}, validación fuera de muestra: {len(oos)}")

# === 6) Ajustar modelo GARCH(1,1) con distribución t (media constante) ===
# Escalamos a porcentajes porque arch espera normalmente datos en porcentaje
train_pct = train * 100.0

model = arch_model(train_pct, mean='Constant', vol='Garch', p=1, q=1, dist='t')
model_fit = model.fit(disp='off', show_warning=False)
print("\n---- Resumen del modelo GARCH(1,1) dist t ----")
print(model_fit.summary())

# AIC / BIC
print(f"AIC: {model_fit.aic:.4f}, BIC: {model_fit.bic:.4f}")

# === 7) Diagnóstico de residuos estandarizados ===
std_resid = model_fit.resid / model_fit.conditional_volatility
std_resid = std_resid.dropna()

print("\nLjung-Box sobre residuos estandarizados (lag=10):")
print(acorr_ljungbox(std_resid, lags=[10], return_df=True))

print("\nLjung-Box sobre residuos estandarizados^2 (lag=10):")
print(acorr_ljungbox(std_resid**2, lags=[10], return_df=True))

# === 8) Forecast (out-of-sample via simulación GARCH) ===
# Parámetros del GARCH ajustado
params = model_fit.params
# Nombres habituales: 'omega', 'alpha[1]', 'beta[1]', 'mu', 'nu'
omega = params.get('omega', None)
alpha = params.get('alpha[1]', params.get('alpha', None))
beta = params.get('beta[1]', params.get('beta', None))
mu = params.get('mu', 0.0)  # media constante en porcentaje
nu = params.get('nu', None) # grados de libertad para t

if any(v is None for v in [omega, alpha, beta, nu]):
    raise RuntimeError("No se pudieron extraer parámetros ω, α, β o ν del modelo ajustado.")

print(f"\nParámetros GARCH: omega={omega:.6g}, alpha={alpha:.6g}, beta={beta:.6g}, mu={mu:.6g}, nu={nu:.4f}")

# Inicializar condiciones con el último dato de train
last_h = (model_fit.conditional_volatility.iloc[-1])**2  # var condicional última fecha (en %^2)
last_r = train_pct.iloc[-1]  # último retorno (en %)

# Función para simular trayectorias GARCH con innovación t-estudent estandarizada
def simulate_garch_paths(n_sim, horizon, omega, alpha, beta, mu, last_h, last_r, nu, random_state=None):
    rng = np.random.default_rng(random_state)
    # factor para estandarizar t: var(t_df) = nu/(nu-2) para nu>2
    if nu <= 2:
        raise ValueError("nu debe ser > 2 para var finita en t-distribution.")
    t_var = nu / (nu - 2.0)
    scale = np.sqrt(t_var)  # para convertir standard_t a var=t_var; dividirás por scale para var=1

    # Arrays: cada fila = simulación, columnas = horizonte
    sim_returns = np.zeros((n_sim, horizon))
    sim_h = np.zeros((n_sim, horizon))

    # condiciones iniciales (seas row vectors)
    h_t = np.full(shape=n_sim, fill_value=last_h)   # var condicional actual repetida
    r_t = np.full(shape=n_sim, fill_value=last_r)   # último retorno en %
    for t in range(horizon):
        # z: t-distribution estandarizada a var=1
        z = rng.standard_t(df=nu, size=n_sim) / scale
        # generar nueva var condicional
        h_new = omega + alpha * (r_t - mu)**2 + beta * h_t
        # generar retorno: r = mu + sqrt(h_new) * z
        r_new = mu + np.sqrt(h_new) * z
        # almacenar
        sim_h[:, t] = h_new
        sim_returns[:, t] = r_new
        # actualizar
        h_t = h_new
        r_t = r_new
    # devolver en unidades de porcentaje (%)
    return sim_returns, sim_h

# Simulaciones forward (usamos el último dato de entrenamiento como base)
sim_returns_pct, sim_h = simulate_garch_paths(n_sim=n_sim, horizon=n_days,
                                              omega=omega, alpha=alpha, beta=beta,
                                              mu=mu, last_h=last_h, last_r=last_r, nu=nu,
                                              random_state=42)

# Convertir returns porcentuales a log-returns (proporciones)
# r_pct es en %, así log-return = r_pct/100
sim_log_returns = sim_returns_pct / 100.0  # dimensiones (n_sim, n_days)

# Precios simulados
P0 = float(prices.iloc[-1].iloc[0])
sim_prices = np.zeros_like(sim_log_returns)
sim_prices[:, 0] = P0 * np.exp(sim_log_returns[:, 0])
for t in range(1, n_days):
    sim_prices[:, t] = sim_prices[:, t-1] * np.exp(sim_log_returns[:, t])

# Compilar forecast (estadísticos)
future_dates = pd.bdate_range(start=prices.index[-1], periods=n_days + 1)[1:]
sim_df = pd.DataFrame(sim_prices, columns=future_dates)
percentiles = [5, 25, 50, 75, 95]
sim_summary = sim_df.quantile(q=np.array(percentiles) / 100.0)

print(f"\nPronóstico (simulación GARCH) - percentiles para los próximos {n_days} días:")
print(sim_summary.T)  # transpuesto para leer por fecha

# Mediana forecast (trayectoria 'esperada' mediana)
median_forecast = sim_summary.loc[0.50].values

# === 9) Gráficas: forecast y simulaciones ===
plt.figure(figsize=(12, 5))
plt.plot(prices.iloc[-100:], label='Histórico (últimos 100 días)')
plt.plot(future_dates, median_forecast, linestyle='--', marker='o', label=f'Forecast mediana (GARCH) {n_days} días', linewidth=2)
plt.fill_between(future_dates, sim_summary.loc[0.05], sim_summary.loc[0.95], color='red', alpha=0.1, label='Intervalo 90%')
plt.fill_between(future_dates, sim_summary.loc[0.25], sim_summary.loc[0.75], color='red', alpha=0.2, label='Intervalo 50%')
plt.title(f'{ticker} - Histórico + Forecast GARCH (simulación)')
plt.xlabel('Fecha'); plt.ylabel('Precio'); plt.legend(); plt.grid(True); plt.show()

# Graficar subset de simulaciones
plt.figure(figsize=(12, 6))
for i in range(min(n_plot_sim, n_sim)):
    plt.plot(future_dates, sim_prices[i, :], color='gray', alpha=0.08)
plt.plot(future_dates, sim_prices.mean(axis=0), color='red', label='Promedio simulaciones', linewidth=2)
plt.title(f'{ticker} - Simulación Monte Carlo GARCH ({n_sim} trayectorias)'); plt.xlabel('Fecha'); plt.ylabel('Precio'); plt.legend(); plt.grid(True); plt.show()

# === 10) Percentiles individuales graficados ===
p5 = sim_summary.loc[0.05].values
p25 = sim_summary.loc[0.25].values
p50 = sim_summary.loc[0.50].values
p75 = sim_summary.loc[0.75].values
p95 = sim_summary.loc[0.95].values

plt.figure(figsize=(12, 6))
plt.plot(future_dates, p50, label='Mediana (50%)', linewidth=2)
plt.plot(future_dates, p5, label='Percentil 5%', linestyle='--')
plt.plot(future_dates, p95, label='Percentil 95%', linestyle='--')
plt.fill_between(future_dates, p5, p95, alpha=0.1)
plt.fill_between(future_dates, p25, p75, alpha=0.15, label='Intervalo 50%')
plt.title(f'{ticker} - Percentiles precios simulados (GARCH)'); plt.xlabel('Fecha'); plt.ylabel('Precio'); plt.legend(); plt.grid(True); plt.show()

# === 11) Métricas de riesgo: VaR / CVaR estimados desde simulaciones ===
# Calcular pérdidas/retornos finales en el horizonte n_days (por ejemplo, día n_days)
# Aquí mostramos VaR y CVaR para el primer día y para el último día del horizonte
for idx, col in enumerate(sim_df.columns):
    # returns simulados diarios (log returns) convertidos desde precios: r = ln(P_t/P_{t-1})
    if idx == 0:
        # primer día: comparar precio simulado día1 con P0
        ret = np.log(sim_df.iloc[:, idx].values / P0)
    else:
        ret = np.log(sim_df.iloc[:, idx].values / sim_df.iloc[:, idx-1])
    VaR_95 = np.percentile(ret, 5)
    CVaR_95 = ret[ret <= VaR_95].mean()
    print(f"\nFecha {col.date()}: VaR(95%)={VaR_95:.5f}, CVaR(95%)={CVaR_95:.5f}")

# === 12) Entropía de la volatilidad condicional reciente (complementaria) ===
hist_vol = (model_fit.conditional_volatility.iloc[-500:]) / 100.0  # en proporciones
vol_bins = pd.cut(hist_vol, bins=20, labels=False)
vol_counts = np.bincount(vol_bins.dropna().astype(int), minlength=20)
vol_dist = vol_counts / vol_counts.sum()
market_entropy = entropy(vol_dist, base=2)
print(f"\nEntropía de mercado (últimos 500 días, sobre volatilidad): {market_entropy:.4f} bits")

# Señal simple basada en desviación estándar
current_vol = hist_vol.iloc[-1]
vol_mean = hist_vol.mean()
vol_std = hist_vol.std()
if current_vol < vol_mean - vol_std:
    signal = "✅ Condición ACEPTABLE (baja volatilidad)"
elif current_vol > vol_mean + vol_std:
    signal = "⚠️ Condición CRÍTICA (alta volatilidad)"
else:
    signal = "🟡 Condición INCIERTA (volatilidad media)"
print(f"Volatilidad actual: {current_vol:.6f}, media: {vol_mean:.6f}, std: {vol_std:.6f} -> {signal}")

# === 13) Validación fuera de muestra (comparar forecast con oos si deseas) ===
# Podemos comparar la mediana simulada del primer día con el retorno observado en oos[0]
if len(oos) > 0:
    # retorno observado (valor escalar)
    observed_return_oos = float(oos.iloc[0].iloc[0])
    # mediana simulada en precios para el primer día
    median_price_day1 = float(sim_summary.loc[0.50].iloc[0])
    # convertir a retorno log relativo a precio actual
    median_ret_day1 = np.log(median_price_day1 / P0)

    print("\nValidación OOS - primer día:")
    print(f"Retorno observado OOS primer día: {observed_return_oos:.6f} / Mediana simulada (log-retorno): {median_ret_day1:.6f}")


print("\n---- FIN del análisis ----")

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pytz

def analizar_activo(symbol, nombre):
    print(f"\nDescargando datos para {nombre} ({symbol})...")
    data = yf.download(tickers=symbol, interval="5m", period="5d")
    data = data.dropna()

    data['log_return'] = np.log(data['Close'] / data['Close'].shift(1))
    window = 20
    threshold = 2

    data['mean'] = data['log_return'].rolling(window=window).mean()
    data['std'] = data['log_return'].rolling(window=window).std()

    data['upper'] = data['mean'] + threshold * data['std']
    data['lower'] = data['mean'] - threshold * data['std']

    data['anomaly'] = (data['log_return'] > data['upper']) | (data['log_return'] < data['lower'])

    if data['anomaly'].any():
        anomalies = data[data['anomaly']].copy()
        last_5_anomalies = anomalies.tail(5)

        print("\nÚltimas 5 anomalías detectadas (hora NY):")
        for idx, row in last_5_anomalies.iterrows():
            if idx.tzinfo is None:
                idx = idx.tz_localize('UTC')
            idx_ny = idx.tz_convert('America/New_York')
            log_return_val = float(row['log_return'])
            print(f"- {idx_ny.strftime('%Y-%m-%d %H:%M:%S')} | Log Return: {log_return_val:.5f}")

        last_anomaly = last_5_anomalies.iloc[-1]
        last_anomaly_time_utc = last_5_anomalies.index[-1]
        if last_anomaly_time_utc.tzinfo is None:
            last_anomaly_time_utc = last_anomaly_time_utc.tz_localize('UTC')
    else:
        print("\nNo se detectaron anomalías.")
        last_anomaly = None

    plt.figure(figsize=(15,6))
    plt.plot(data.index, data['log_return'], label='Log Return', color='black', alpha=0.5)
    plt.plot(data.index, data['upper'], label='Upper Threshold', color='red', linestyle='--')
    plt.plot(data.index, data['lower'], label='Lower Threshold', color='green', linestyle='--')
    plt.scatter(data[data['anomaly']].index, data[data['anomaly']]['log_return'], color='orange', label='Anomalies', zorder=5)

    if last_anomaly is not None:
        log_ret = float(last_anomaly['log_return'])
        upper = float(last_anomaly['upper'])
        direction = "por arriba" if log_ret > upper else "por debajo"
        plt.annotate(
            f"Rendimiento logarítmico {direction} de 2 desviaciones estándar\nBuscar posible corrección",
            xy=(last_anomaly_time_utc, log_ret),
            xytext=(last_anomaly_time_utc, log_ret + 0.002),
            arrowprops=dict(facecolor='red', shrink=0.05),
            fontsize=9,
            backgroundcolor='yellow'
        )

    plt.title(f'Anomalías en los Rendimientos Logarítmicos - {nombre} ({symbol})')
    plt.xlabel('Fecha')
    plt.ylabel('Log Return')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


def menu():
    intro_text = """
    *** Indicador Estadístico de Anomalías en Rendimientos Logarítmicos ***

    Este programa no es una estrategia de trading, sino un indicador que mide
    condiciones estadísticas del mercado mediante el análisis de los rendimientos
    logarítmicos intradía del activo seleccionado.

    Los rendimientos logarítmicos son efectivos para detectar posibles reversiones
    o correcciones porque reflejan cambios proporcionales y simétricos en los precios,
    lo que facilita identificar movimientos extremos comparados con la volatilidad
    histórica reciente.

    IMPORTANTE:
    - No se deben tomar decisiones de compra o venta basándose únicamente en este indicador.
    - Este análisis debe complementarse con otras herramientas y análisis técnico.
    """

    print(intro_text)

    while True:
        print("\nSeleccione un activo para analizar:")
        print("1. S&P 500 (^GSPC)")
        print("2. NASDAQ 100 (^NDX)")
        print("3. Dow Jones 30 (^DJI)")
        print("4. US100 (Nasdaq 100) (^NDX)")
        print("5. EUR/USD (EURUSD=X)")
        print("6. GBP/USD (GBPUSD=X)")
        print("7. USD/JPY (JPY=X)")
        print("8. USD/CAD (CAD=X)")
        print("9. Salir")

        opcion = input("Ingrese opción (1-9): ").strip()

        if opcion == '1':
            analizar_activo("^GSPC", "S&P 500")
        elif opcion == '2':
            analizar_activo("^NDX", "NASDAQ 100")
        elif opcion == '3':
            analizar_activo("^DJI", "Dow Jones 30")
        elif opcion == '4':
            analizar_activo("^NDX", "US100 (Nasdaq 100)")
        elif opcion == '5':
            analizar_activo("EURUSD=X", "EUR/USD")
        elif opcion == '6':
            analizar_activo("GBPUSD=X", "GBP/USD")
        elif opcion == '7':
            analizar_activo("JPY=X", "USD/JPY")
        elif opcion == '8':
            analizar_activo("CAD=X", "USD/CAD")
        elif opcion == '9':
            print("Saliendo...")
            break
        else:
            print("Opción inválida. Intente nuevamente.")


if __name__ == "__main__":
    menu()